# Train results — R3

**Strategy R3: adaptive scope (slot -> day -> plan)**

This notebook reproduces the `R3` column of the **Train results** tab in `Results.xlsx`, end to end. It:

1. Runs the repository's own analyzer (`scripts/analyze_repair_strategies.py`) on the `R3` training run (`experiments/subset_train-r3-new`).
2. Reads the analyzer's `overall_summary.csv` and `mechanism_summary.csv`.
3. Derives the **common-core** metrics (shared by every strategy) and the **R3-specific** metrics.
4. Writes the exact table to `Train_results_R3.xlsx`.

Everything is read-only analysis of an existing run: **no experiments, no model calls, no benchmark scoring.** The outputs below are produced by actually executing the cells.

## Step 0 — Setup and run the analyzer
We run from the repository root and point the analyzer at this strategy's training run. The analyzer writes its tables under a scratch output directory.

In [1]:
import subprocess, sys, os, csv
from pathlib import Path

# Resolve the repository root (the folder that contains 'scripts/' and 'experiments/').
ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists(), 'Run this from inside the repo.'
NB_DIR = Path.cwd()  # folder containing this notebook (before we chdir to repo root)
os.chdir(ROOT)
PY = sys.executable  # the kernel's interpreter (the project venv)

STRATEGY = 'R3'
RUN_DIR = 'experiments/subset_train-r3-new'
# Analyzer output is written here, NEXT TO THIS NOTEBOOK (not a temp dir),
# so you can open overall_summary.csv / mechanism_summary.csv yourself.
OUT_DIR = str(NB_DIR / 'analysis_output')

assert Path(RUN_DIR).is_dir(), f'Missing run folder: {RUN_DIR}'
print('Repo root :', ROOT)
print('Strategy  :', STRATEGY)
print('Run folder:', RUN_DIR)
print('Analyzer output dir:', OUT_DIR)

Repo root : /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-
Strategy  : R3
Run folder: experiments/subset_train-r3-new
Analyzer output dir: /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output


In [2]:
# Run the repo's analyzer for this one strategy. This only reads the saved run.
cmd = [PY, 'scripts/analyze_repair_strategies.py',
       '--run', f'{STRATEGY}={RUN_DIR}',
       '--output-dir', OUT_DIR, '--no-figures']
print('$', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout.strip()[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'analyzer failed'
print('\nAnalyzer exit code:', res.returncode)

$ /Users/enruifung/.pyenv/versions/3.12.8/bin/python3 scripts/analyze_repair_strategies.py --run R3=experiments/subset_train-r3-new --output-dir /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/analysis_output --no-figures


R3: source=experiments/subset_train-r3-new, declared/evidenced queries=60, attempt queries=57, proposals=325, matrix cells=720

Analyzer exit code: 0


## Step 1 — Load the analyzer's summary tables

> **Where do these CSVs come from?** They are **not** pre-existing files in the
> repo. **Step 0 above created them** by running the analyzer. They are written to
> `train_result_analysis/analysis_output/<STRATEGY>/` (printed by Step 0). If you
> look before running Step 0, the folder will not exist yet — that's expected.

`overall_summary.csv` holds counts/means per run; `mechanism_summary.csv` holds the strategy-specific mechanism metrics.

In [3]:
def load_summary(path):
    d = {}
    with open(path) as f:
        for row in csv.DictReader(f):
            d[row['metric']] = row
    return d

overall = load_summary(f'{OUT_DIR}/{STRATEGY}/overall_summary.csv')
mech = load_summary(f'{OUT_DIR}/{STRATEGY}/mechanism_summary.csv')
print('overall_summary metrics:', len(overall))
print('mechanism_summary metrics:', len(mech))
print('\nmechanism metrics available:')
for k, v in mech.items():
    print(f'  {k}: count={v["count"]} valid_n={v["valid_n"]} rate={v["rate"]}')

overall_summary metrics: 85
mechanism_summary metrics: 8

mechanism metrics available:
  next_scope_widens: count=107 valid_n=268 rate=0.399253731343284
  noop_followed_by_widening: count=62 valid_n=134 rate=0.462686567164179
  slot_proposal_fixing_any: count=12 valid_n=111 rate=0.108108108108108
  slot_proposal_regressing_any: count=6 valid_n=111 rate=0.0540540540540541
  day_proposal_fixing_any: count=4 valid_n=108 rate=0.037037037037037
  day_proposal_regressing_any: count=3 valid_n=108 rate=0.0277777777777778
  plan_proposal_fixing_any: count=2 valid_n=106 rate=0.0188679245283019
  plan_proposal_regressing_any: count=7 valid_n=106 rate=0.0660377358490566


## Step 2 — Common-core metrics (shared by every strategy)
These come straight from `overall_summary.csv`. 'Confirmed unchanged attempts' is shown as `N (pct%)` of total attempts, matching the R1 template already in the spreadsheet.

In [4]:
def iv(metric):
    return int(float(overall[metric]['value']))

queries              = iv('declared_queries')
repair_attempts      = iv('total_attempts')
personal_attempts    = iv('personal_attempts')
commonsense_attempts = iv('commonsense_attempts')
unchanged            = iv('unchanged_confirmed')
unchanged_pct        = round(unchanged / repair_attempts * 100, 2)
personal_fixes       = iv('personal_fixed_count_sum')
personal_regressions = iv('personal_newly_violated_count_sum')

core = [
    ('Queries', queries),
    ('Repair attempts', repair_attempts),
    ('Personal repair attempts', personal_attempts),
    ('Commonsense repair attempts', commonsense_attempts),
    ('Confirmed unchanged attempts', f'{unchanged} ({unchanged_pct}%)'),
    ('Personal constraint fixes', personal_fixes),
    ('Personal regressions', personal_regressions),
]
for k, v in core:
    print(f'  {k:40} {v}')

  Queries                                  60
  Repair attempts                          325
  Personal repair attempts                 145
  Commonsense repair attempts              180
  Confirmed unchanged attempts             161 (49.54%)
  Personal constraint fixes                18
  Personal regressions                     2


## Step 3 — R3-specific metrics
These describe R3's own mechanism, read from `mechanism_summary.csv`. Each is formatted `count / valid_n (rate%)`.

In [5]:
SPECIFIC = [('Attempts where scope widened next', 'next_scope_widens', 'count_rate'), ('No-op attempts followed by widening', 'noop_followed_by_widening', 'count_rate'), ('Slot-scope proposals fixing any constraint', 'slot_proposal_fixing_any', 'count_rate'), ('Slot-scope proposals regressing any', 'slot_proposal_regressing_any', 'count_rate'), ('Day-scope proposals fixing any', 'day_proposal_fixing_any', 'count_rate'), ('Day-scope proposals regressing any', 'day_proposal_regressing_any', 'count_rate'), ('Plan-scope proposals fixing any', 'plan_proposal_fixing_any', 'count_rate'), ('Plan-scope proposals regressing any', 'plan_proposal_regressing_any', 'count_rate')]

def fmt_count_rate(row):
    c = int(float(row['count']))
    n = int(float(row['valid_n']))
    r = float(row['rate']) * 100
    return f'{c} / {n} ({r:.1f}%)'

specific = []
for label, metric, kind in SPECIFIC:
    if metric not in mech:
        specific.append((label, 'n/a (metric not produced for this run)'))
        continue
    row = mech[metric]
    specific.append((label, fmt_count_rate(row) if kind == 'count_rate' else row['rate']))
for k, v in specific:
    print(f'  {k:60} {v}')

  Attempts where scope widened next                            107 / 268 (39.9%)
  No-op attempts followed by widening                          62 / 134 (46.3%)
  Slot-scope proposals fixing any constraint                   12 / 111 (10.8%)
  Slot-scope proposals regressing any                          6 / 111 (5.4%)
  Day-scope proposals fixing any                               4 / 108 (3.7%)
  Day-scope proposals regressing any                           3 / 108 (2.8%)
  Plan-scope proposals fixing any                              2 / 106 (1.9%)
  Plan-scope proposals regressing any                          7 / 106 (6.6%)


## Step 4 — Assemble the full table

In [6]:
rows = [('Metric', f'{STRATEGY} training result')]
rows += [(k, v) for k, v in core]
rows += [('', '')]  # spacer between core and strategy-specific
rows += [(k, v) for k, v in specific]

for k, v in rows:
    print(f'{k:60} | {v}')

Metric                                                       | R3 training result
Queries                                                      | 60
Repair attempts                                              | 325
Personal repair attempts                                     | 145
Commonsense repair attempts                                  | 180
Confirmed unchanged attempts                                 | 161 (49.54%)
Personal constraint fixes                                    | 18
Personal regressions                                         | 2
                                                             | 
Attempts where scope widened next                            | 107 / 268 (39.9%)
No-op attempts followed by widening                          | 62 / 134 (46.3%)
Slot-scope proposals fixing any constraint                   | 12 / 111 (10.8%)
Slot-scope proposals regressing any                          | 6 / 111 (5.4%)
Day-scope proposals fixing any                               

## Step 5 — Write the exact table to `Train_results_R3.xlsx`

In [7]:
import openpyxl
from openpyxl.styles import Font

wb = openpyxl.Workbook()
ws = wb.active
ws.title = f'{STRATEGY} train results'

for r, (k, v) in enumerate(rows, start=1):
    ws.cell(r, 1, k)
    ws.cell(r, 2, v)

# bold the header row
ws.cell(1, 1).font = Font(bold=True)
ws.cell(1, 2).font = Font(bold=True)
ws.column_dimensions['A'].width = 52
ws.column_dimensions['B'].width = 40

out_xlsx = NB_DIR / f'Train_results_{STRATEGY}.xlsx'
wb.save(out_xlsx)
print('Wrote', out_xlsx)
print('Rows written:', len(rows))

Wrote /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/Train_results_R3.xlsx
Rows written: 17


## Step 6 — Read the file back to confirm it saved correctly

In [8]:
chk = openpyxl.load_workbook(out_xlsx, data_only=True)
cs = chk.active
print('Sheet:', cs.title, '| dims:', cs.dimensions)
print()
for row in cs.iter_rows(values_only=True):
    if any(c not in (None, '') for c in row):
        print(f'{str(row[0]):52} | {row[1]}')

Sheet: R3 train results | dims: A1:B17

Metric                                               | R3 training result
Queries                                              | 60
Repair attempts                                      | 325
Personal repair attempts                             | 145
Commonsense repair attempts                          | 180
Confirmed unchanged attempts                         | 161 (49.54%)
Personal constraint fixes                            | 18
Personal regressions                                 | 2
Attempts where scope widened next                    | 107 / 268 (39.9%)
No-op attempts followed by widening                  | 62 / 134 (46.3%)
Slot-scope proposals fixing any constraint           | 12 / 111 (10.8%)
Slot-scope proposals regressing any                  | 6 / 111 (5.4%)
Day-scope proposals fixing any                       | 4 / 108 (3.7%)
Day-scope proposals regressing any                   | 3 / 108 (2.8%)
Plan-scope proposals fixing any          